# Кросс-валидация rubert-tiny2 на всей разметке «Повестей Белкина»

Вся своя разметка (694 слова, раунды 1-3) делится на 5 частей так, что слово целиком попадает в одну часть. Модель учится на четырёх частях и предсказывает пятую, которую не видела. Так каждое слово получает честное предсказание, и метрики считаются по 694 словам вместо 196.

Перед запуском: **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**. Дальше выполняй ячейки по порядку. Каждый сид запускается в своей ячейке, и файлы скачиваются сразу после неё: если Colab потеряет среду, уже скачанное не пропадёт.

In [ ]:
!nvidia-smi -L || echo "GPU не найден: обучение пойдёт на CPU и займёт намного дольше" 

## 1. Код и зависимости

In [ ]:
!rm -rf archaic-words
!git clone https://github.com/borisovv1905-art/archaic-words.git
%cd archaic-words
!pip install -q -e . transformers

## 2. Публичные данные (Библия, Zenodo 7419194)
Ячейка пробует скачать архив сама. Если не получилось, скачай `lexical_complexity.zip` на странице https://zenodo.org/records/7419194 и загрузи его через значок папки слева, в корень `/content/archaic-words`.

In [ ]:
import os, zipfile, subprocess, glob
os.makedirs("data/raw", exist_ok=True)
if not glob.glob("data/raw/lexical_complexity/word_complexity_ru_train.tsv"):
    if not os.path.exists("lexical_complexity.zip"):
        subprocess.run(["wget", "-q", "-O", "lexical_complexity.zip",
                        "https://zenodo.org/records/7419194/files/lexical_complexity.zip?download=1"])
    assert os.path.exists("lexical_complexity.zip") and os.path.getsize("lexical_complexity.zip") > 10000, \
        "Архив не скачался: загрузи lexical_complexity.zip вручную (см. текст выше)"
    zipfile.ZipFile("lexical_complexity.zip").extractall("data/raw")
print(glob.glob("data/raw/**/*.tsv", recursive=True))

In [ ]:
# быстрая проверка, что всё на месте
!ls data/raw/lexical_complexity data/annotation
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())" 

## Быстрая проверка (около минуты)
Прогон на крошечных выборках: убедиться, что всё работает, до того как ждать полчаса. Цифры в выводе ничего не значат. Если здесь красная ошибка, дальше не иди и пришли мне её скриншот.

In [ ]:
!python -m archword.crossval --raw data/raw --setting bible_then_own --smoke --out results/smoke.json

## 3. Сид 0
Этап Библии обучается один раз, потом пять раз дообучается на своей разметке. По моей оценке, это порядка 20-30 минут на T4. В выводе появляются строки «сид 0, часть N: готово».

In [ ]:
!python -m archword.crossval --raw data/raw --setting bible_then_own --seeds 1 --first-seed 0 --out results/crossval_bible_then_own_s0.json
from google.colab import files
files.download("results/crossval_bible_then_own_s0.json")
files.download("results/crossval_bible_then_own_s0.predictions.csv")

## 4. Сид 1
Второй независимый запуск: так видно, насколько результат зависит от случайности обучения. Запускай после того, как скачались файлы сида 0.

In [ ]:
!python -m archword.crossval --raw data/raw --setting bible_then_own --seeds 1 --first-seed 1 --out results/crossval_bible_then_own_s1.json
from google.colab import files
files.download("results/crossval_bible_then_own_s1.json")
files.download("results/crossval_bible_then_own_s1.predictions.csv")

## 5. По желанию: без Библии
Тот же разбор, но модель учится только на своей разметке. Показывает, сколько даёт этап Библии. Быстрее предыдущих (порядка 10 минут).

In [ ]:
!python -m archword.crossval --raw data/raw --setting own --seeds 1 --first-seed 0 --out results/crossval_own_s0.json
from google.colab import files
files.download("results/crossval_own_s0.json")
files.download("results/crossval_own_s0.predictions.csv")